# ARC_ATLAS v4 Slice-Block Resume Fine-Tune

This notebook resumes from the current best slice-block checkpoint and changes only the training objective/thresholding behavior. The model architecture is unchanged, so the existing weights load directly.

Resume source:
`runs/20260420_120656_slice_blocks/callbacks/best_slice_block.weights.h5`

Fine-tune changes:
- lower decision threshold to `0.10`
- sweep lower thresholds from `0.03` upward
- downweight empty slices
- upweight lesion-containing slices
- compute Dice/Tversky only on lesion slices
- add a positive top-k term to push missed lesion voxels upward
- flip Tversky toward recall with `alpha=0.30`, `beta=0.70`


In [ ]:
from pathlib import Path
import sys
import time

candidates = [
    Path.cwd(),
    Path.cwd() / "ARC_ATLAS_Combined" / "ARC_ATLAS_Train_v4",
    Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4"),
]
PROJECT_ROOT = next(
    (p for p in candidates if (p / "src" / "training_v2_slice_blocks.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate ARC_ATLAS_Train_v4/src/training_v2_slice_blocks.py")

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import training_v2_slice_blocks as seg

TRAIN_DIR = PROJECT_ROOT / "data" / "splits" / "90_10_random" / "train"
SOURCE_RUN = PROJECT_ROOT / "runs" / "20260420_120656_slice_blocks"
INITIAL_WEIGHTS = SOURCE_RUN / "callbacks" / "best_slice_block.weights.h5"
RUN_DIR = PROJECT_ROOT / "runs" / f"{time.strftime('%Y%m%d_%H%M%S')}_slice_blocks_resume_smalllesion"

if not INITIAL_WEIGHTS.exists():
    raise FileNotFoundError(f"Resume checkpoint not found: {INITIAL_WEIGHTS}")

cfg = seg.SliceBlockTrainingConfig(
    DATA_DIR=TRAIN_DIR,
    IMAGES_DIR=TRAIN_DIR / "t1",
    MASKS_DIR=TRAIN_DIR / "masks",
    MANIFEST_PATH=TRAIN_DIR / "manifest.csv",
    MODEL_DIR=RUN_DIR / "models",
    CALLBACKS_DIR=RUN_DIR / "callbacks",
    INITIAL_WEIGHTS_PATH=INITIAL_WEIGHTS,
    TARGET_SHAPE=(192, 224, 192),
    RESAMPLE_TO_TARGET=False,
    SLICE_AXIS=2,
    BLOCK_DEPTH=3,
    SLICE_STRIDE=1,
    TOTAL_EPOCHS=40,
    INITIAL_LR=5e-5,
    MIN_LR=5e-7,
    BASE_FILTERS=8,
    UNET_DEPTH=4,
    POSITIVE_WEIGHT=50.0,
    BCE_WEIGHT=0.40,
    DICE_WEIGHT=0.45,
    FOCAL_TVERSKY_WEIGHT=0.15,
    TVERSKY_ALPHA=0.30,
    TVERSKY_BETA=0.70,
    FOCAL_TVERSKY_GAMMA=1.33,
    LESION_SLICE_WEIGHT=3.0,
    EMPTY_SLICE_WEIGHT=0.20,
    DICE_ON_LESION_SLICES_ONLY=True,
    POSITIVE_TOPK_WEIGHT=0.10,
    POSITIVE_TOPK_FRACTION=0.20,
    DECISION_THRESHOLD=0.10,
    VAL_THRESHOLD_SWEEP=(0.03, 0.05, 0.075, 0.10, 0.125, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50),
    WHOLE_BRAIN_VAL_EVERY_N_EPOCHS=1,
    WHOLE_BRAIN_VAL_MAX_CASES=None,
    SAVE_VAL_PREDICTIONS=True,
    NUM_VAL_PREDICTIONS=5,
    FIT_VERBOSE=2,
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Resume checkpoint: {INITIAL_WEIGHTS}")
print(f"New run dir: {RUN_DIR}")
print(f"Input per brain: (num_slices, {cfg.input_shape[0]}, {cfg.input_shape[1]}, {cfg.input_shape[2]})")
print(f"Decision threshold: {cfg.DECISION_THRESHOLD}")
print(f"Threshold sweep: {cfg.VAL_THRESHOLD_SWEEP}")
print(f"Epoch log CSV: {cfg.CALLBACKS_DIR / 'training_log.csv'}")
print(f"Whole-brain validation summary: {cfg.CALLBACKS_DIR / 'whole_val_summary.jsonl'}")


In [ ]:
# Verify the checkpoint loads before starting the fine-tune.
model = seg.build_slice_block_model(cfg)
model.load_weights(str(INITIAL_WEIGHTS))
print(f"Loaded weights into model with {model.count_params():,} parameters")
del model
seg.tf.keras.backend.clear_session()


In [ ]:
# Sanity check one full-geometry case under the resume config.
cases = seg.load_cases(cfg)
image, mask, _ = seg.load_case_arrays(cases[0], cfg)
x, y = seg.make_slice_blocks(image, mask, cfg)
print(f"Cases: {len(cases)}")
print(f"Prepared brain: image={image.shape}, mask={mask.shape}")
print(f"One-brain batch: x={x.shape}, y={y.shape}")
print(f"Lesion voxels in sanity case: {int(y.sum())}")


In [ ]:
# Launch the resume fine-tune. Epoch numbers here are phase-local; epoch 0 means first fine-tune epoch after loading best weights.
history = seg.train_slice_block_model(cfg)


In [ ]:
# Review the fine-tune logs.
import json
import pandas as pd

train_log = cfg.CALLBACKS_DIR / "training_log.csv"
whole_val_log = cfg.CALLBACKS_DIR / "whole_val_summary.jsonl"

if train_log.exists():
    display(pd.read_csv(train_log).tail(10))
else:
    print(f"Training log not found yet: {train_log}")

if whole_val_log.exists():
    rows = [json.loads(line) for line in whole_val_log.read_text().splitlines() if line.strip()]
    display(pd.DataFrame(rows).tail(10))
else:
    print(f"Whole-brain validation log not found yet: {whole_val_log}")
